<a href="https://colab.research.google.com/github/nyp-sit/it3103-2025s1/blob/main/week14_RL/shift-demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Setting Up

In [1]:
%%capture
%pip install -U transformers==4.52.1
%pip install -U datasets
%pip install -U accelerate
%pip install -U peft
%pip install -U trl
%pip install -U bitsandbytes

In [2]:
from huggingface_hub import login
import os

hf_token = os.environ.get("HF_TOKEN")
login(hf_token)

## Loading the model and tokenizer

In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch


In [4]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=False,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

## GPT-2

In [ ]:
from transformers import AutoTokenizer, GPT2LMHeadModel

tokenizer = AutoTokenizer.from_pretrained('gpt2-large')
model = GPT2LMHeadModel.from_pretrained('gpt2-large')

math_question = "What is the coefficient of $x^2y^6$ in the expansion of $\left(\frac{3}{5}x-\frac{y}{2}\right)^8$? Express your answer as a common fraction."
input_ids = tokenizer(math_question, return_tensors='pt').input_ids
output = model.generate(input_ids, do_sample=True, max_length=2400, top_p=0.95, top_k=0)
print("Output:\n" + 100 * '-')
print(tokenizer.decode(output[0], skip_special_tokens=True))
print("" + 100 * '-')

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/666 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.25G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Output:
----------------------------------------------------------------------------------------------------
ight)^8$? Express your answer as a common fraction.

On your own, before using logarithms, you will need to present the answers to the usual questions: How much numerator is "3" and "5"? What is the decimal digit (0-9)? What is the notation for % (i.e. 0-9). What is the decimal digit ( 0-9)? How many units from $3$ to $5$ would it take to satisfy this series?

Next, give a formula of inverse logarithm of "factor", like this:

If $x$ is 0, $Log(x) = 0$, then $x = \frac{x}{2}$, and $x^2y^6 = (x^2y)^2$.

If $x$ is -1, $Log(x) = 0$, then $x = (x^2y)^2 = (x^2y)^2$.

If $x$ is > -1, $Log(x) = -1$, then $x = (x^2y)^2 = (x^2y)^2$.

If $x$ is <= 0, $Log(x) = 1$ and $x = (x^2y)^2 = (x^2y)^2$, then $x^2y^2 = 1$, and $x^2y^6 = 1$.

If $x$ is > 0, $Log(x) = 0$ and $x = (x^2y)^2 = (x^2y)^2$, then $x^2y^6 = 1$, and $x^2y^8 = 1$.

If $x$ is < 0, $Log(x) = 0$ and $x = (x^2y)^2 = (x^2y)^2$, then 

## DeepSeek-R1

In [6]:
# Load tokenizer & model

model_dir = "deepseek-ai/DeepSeek-R1-0528-Qwen3-8B"

tokenizer = AutoTokenizer.from_pretrained(model_dir, use_fast=True)

model = AutoModelForCausalLM.from_pretrained(
    model_dir,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    trust_remote_code=True
)

model.config.use_cache = False
model.config.pretraining_tp = 1

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/859 [00:00<?, ?B/s]

Unrecognized keys in `rope_scaling` for 'rope_type'='yarn': {'attn_factor'}


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00002-of-000002.safetensors:   0%|          | 0.00/7.77G [00:00<?, ?B/s]

model-00001-of-000002.safetensors:   0%|          | 0.00/8.61G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

## Loading and processing the dataset

In [8]:
EOS_TOKEN = tokenizer.eos_token  # Must add EOS_TOKEN

def formatting_prompts_func(examples):
    inputs = examples["input"]
    outputs = examples["output"]
    texts = []
    for question, response in zip(inputs, outputs):
        # Remove the "Q:" prefix from the question
        question = question.replace("Q:", "")

        # Append the EOS token to the response if it's not already there
        if not response.endswith(tokenizer.eos_token):
            response += tokenizer.eos_token

        text = train_prompt_style.format(question, response)
        texts.append(text)
    return {"text": texts}

## Model inference

In [11]:
inference_prompt_style = """
Please answer with one of the options in the bracket. Write reasoning in between <analysis></analysis>. Write the answer in between <answer></answer>.

### Question:
{}

### Response:
<analysis>
"""

In [ ]:
question = math_question


inputs = tokenizer(
    [inference_prompt_style.format(question) + tokenizer.eos_token],
    return_tensors="pt"
).to("cuda")

outputs = model.generate(
    input_ids=inputs.input_ids,
    attention_mask=inputs.attention_mask,
    max_new_tokens=2400,
    eos_token_id=tokenizer.eos_token_id,
    use_cache=True,
)
response = tokenizer.batch_decode(outputs, skip_special_tokens=True)
print(response[0].split("### Response:")[1])

Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.



<analysis>
<think>
I need to find the coefficient of \( x^2 y^6 \) in the expansion of \( \left( \frac{3}{5} x - \frac{y}{2} \right)^8 \). I'll use the binomial theorem, which states that \( (a + b)^n = \sum_{k=0}^{n} \binom{n}{k} a^{n-k} b^k \). Here, a is \( \frac{3}{5} x \) and b is \( -\frac{y}{2} \), and n is 8.

The general term in the expansion is \( \binom{8}{k} \left( \frac{3}{5} x \right)^{8-k} \left( -\frac{y}{2} \right)^k \).

I need the term where the power of x is 2 and the power of y is 6. So, in the general term, the exponent of x is 8 - k, and the exponent of y is k, because b is \( -\frac{y}{2} \), so when we raise it to k, we get y^k, but there's a coefficient.

Set up the equation for the exponents. For x^2, 8 - k = 2, so k = 8 - 2 = 6.

For y^6, k = 6, since y^k and k is the exponent.

So, when k=6, the term is \( \binom{8}{6} \left( \frac{3}{5} x \right)^{8-6} \left( -\frac{y}{2} \right)^6 \).

Simplify that. First, \( \binom{8}{6} = \binom{8}{2} \) because binom